# Was our earthquake forecast wrong — or were we just unlucky?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2FAI4EPS%2FEPS88_PyEarth&branch=main&urlpath=lab%2Ftree%2FEPS88_PyEarth%2Fdocs/notebooks%2F08_wrong_or_unlucky.ipynb).

You have already fitted Gutenberg-Richter to California's small earthquakes and used the line to
predict how many magnitude-7 events the state should expect in thirty-six years. The line said
about one and a half. Five happened. You were asked to write down whether you thought your model
had failed, and the honest answer was that you could not tell — because 1.6 and 5 are two bare
numbers, and two bare numbers cannot be compared.

What is missing is a sense of how far that 1.6 would have moved if the earthquakes had fallen
slightly differently. Today you build it, out of nothing but the data you already have. You will
practise on something slower — a hundred and twenty-six years of sea level in San Francisco Bay,
where the question is whether the water is rising at all — and then take the same six lines back
to the earthquakes and settle the argument.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by an empty cell. Fill them all in, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Say how fast San Francisco Bay is rising, with a range rather than a number, and
say whether the rise could be zero. Then say whether California's five large earthquakes are
evidence that a Gutenberg-Richter forecast is broken, or the kind of run of luck a working
forecast produces anyway — and know which of those two questions your interval answered.

**The skills.** Resampling: `table.sample(n, replace=True)` draws a new dataset out of the one you
have, `np.percentile` turns a thousand answers into an interval, and a `for` loop puts the two
together. You will put an error bar on a fitted slope, which is something a single call to
`LinearRegression` will never give you.

**The four questions this week works through:**

1. Could the Bay's rise be zero?
2. How much water is that by 2100?
3. Was the forecast wrong, or were we unlucky?
4. What did we assume without saying so?

**Nine places where you write something: six in class, three at home.** Each one is headed
*Your turn*, with an empty cell under it.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load(url, cached):
    """Read one live source; fall back to the copy stored with the course."""
    try:
        return pd.read_csv(url)
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + cached)

sea = load("https://api.tidesandcurrents.noaa.gov/api/prod/datagetter?product=monthly_mean&station=9414290&datum=STND&units=metric&time_zone=GMT&format=csv&begin_date=19000101&end_date=20251231",
           "week08_sf_sea_level_1900_2025.csv")
sea.columns = sea.columns.str.strip()      # the NOAA file pads its column names with spaces
sea["year"] = sea["Year"] + (sea["Month"] - 0.5) / 12   # 1903.5 means the middle of 1903
sea["sea_mm"] = sea["MSL"] * 1000                       # the file is in metres, we want mm

FDSN = "https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&orderby=time-asc"
CA_BOX = "&minlatitude=32&maxlatitude=42&minlongitude=-125&maxlongitude=-114"

quakes = load(FDSN + "&starttime=1990-01-01&endtime=2026-01-01"
                     "&minmagnitude=3.5&maxmagnitude=6.9" + CA_BOX,
              "week08_ca_1990_2026_M3.5-6.9.csv")
big = load(FDSN + "&starttime=1918-01-01&endtime=2026-01-01&minmagnitude=7.0" + CA_BOX,
           "week08_ca_1918_2026_M7.csv")

print("sea level:", sea.shape, " small earthquakes:", quakes.shape, " large ones:", big.shape)

## Could the Bay's rise be zero?

A tide gauge is a float in a stilling well, bolted to a pier, writing down where the water sits.
The one in San Francisco — NOAA station 9414290 — has been doing that since before anyone thought
sea level was a question, and NOAA publishes a monthly mean from it. That is what `sea` holds: one
monthly mean per month from 1900 to 2025, in the `MSL` column, which the
setup cell turned into millimetres in `sea_mm`. The setup cell printed how many rows that is, and
it is short of the 1,512 months those 126 years would hold: the
gauge has gaps, and the most recent of them are new enough that NOAA may yet fill them in.

Two warnings about what that column is. The zero is arbitrary: the heights are quoted against
*station datum*, a mark on the pier chosen for convenience, so the height itself means nothing and
only the *change* does. And a tide gauge measures the sea against the land it is bolted to, so
what it records is **relative** sea level: if the pier were sinking, the water would appear to
rise by exactly the same amount.

In [ ]:
plt.scatter(sea["year"], sea["sea_mm"], s=2)
plt.xlabel("year")
plt.ylabel("monthly mean sea level (mm above station datum)")
plt.title(f"San Francisco, {len(sea):,} monthly means")
plt.show()

There is a rise in there, and a great deal of noise on top of it: winter storms, El Niño years,
the seasons. A straight line through it is the regression you already know, with `year` as the one
input column and `sea_mm` as the thing to predict. Its slope is millimetres per year.

In [ ]:
fit = LinearRegression().fit(sea[["year"]], sea["sea_mm"])
slope = fit.coef_[0]

print("slope:", round(slope, 3), "mm per year")
print("R squared:", round(fit.score(sea[["year"]], sea["sea_mm"]), 3))

1.965 millimetres a year, and an R squared of 0.544 — the line explains about
half of what the gauge did, and the rest is the month-to-month weather and ocean variability the
line cannot see. Now ask the question that matters: **could
the answer be zero?** The number 1.965 does not answer it. It has nothing after it.

Here is the obvious first move. If the record were telling us something stable, then any big piece
of it should give roughly the same slope. So cut it up and look.

### ✏️ Your turn 1

Cut the record into five twenty-five-year pieces and fit the same line to each one on its own.

Loop over `[1900, 1925, 1950, 1975, 2000]`. For each `start`, keep the rows with
`(sea["year"] >= start) & (sea["year"] < start + 25)`, fit `LinearRegression` to that piece exactly
as the cell above did, and append the slope to a list. Print each piece's start year, how many
months it holds, and its slope.

**Use these names**, because the self-check looks for them: `chunk_slopes`.

In [ ]:
# ← your answer here


assert chunk_slopes[-1] > chunk_slopes[0] + 1, \
    "the last piece should be more than 1 mm/yr steeper than the first"
assert 3 < max(chunk_slopes) - min(chunk_slopes) < 5, \
    "the five slopes should span about 4 mm/yr — check the filter"
print("✓ the record cut five ways — slopes from", round(min(chunk_slopes), 2),
      "to", round(max(chunk_slopes), 2), "mm/yr, against", round(slope, 2), "for the whole record")

Five pieces of the same record, and the slopes run from 0.11 to
4.14 mm/yr. The first quarter of the twentieth century says the Bay was barely
moving; the last twenty-five years say four millimetres a year.

That is honest but useless, for two separate reasons. First, two different things are mixed
together in that spread and staring at five numbers cannot separate them: how much a
twenty-five-year slope wobbles by chance, and whether the rise has genuinely sped up. Second — and
this is the fatal one — we did not ask how much a *twenty-five-year* slope wobbles. We asked how
much the 126-year slope wobbles, and each of those fits threw away four fifths of
the data. Five numbers also cannot tell you what "95% of the time" means.

What we want is the whole record's worth of answer, many times over. We only have one record. So
we make more of it out of the one we have.

## How do you get an error bar out of one record?

The trick has a name and one line of code. **Bootstrap:** *Ask the data the same question a
thousand times, using a different random slice of itself each time.*

The slice is drawn **with replacement**: pick a row at random, write it down, put it back, and
pick again, until you have as many rows as you started with. Some rows get picked twice, some not
at all — and that is the whole point, because that is what a second run of history would have
looked like. Watch it happen on eight numbers first.

In [ ]:
np.random.seed(88)
eight = pd.DataFrame({"height": [1, 2, 3, 4, 5, 6, 7, 8]})

print("the original:", list(eight["height"]))
print("one resample:", list(eight.sample(8, replace=True)["height"]))
print("another:     ", list(eight.sample(8, replace=True)["height"]))

Repeats in one, holes in the other, same length as the original. `np.random.seed` is there for the
same reason as in the probability week: it makes the randomness repeatable, so your numbers and
your neighbour's match.

Now do it to the tide gauge. One resample of `sea` is as many months as the record holds, drawn
with replacement from those same months; fit the line to that and you get a slope that is nearly,
but not quite, 1.965. Do it 2,000 times and you have 2,000 slopes.

Notice which way round this is. When you simulated earthquake times you started from a
distribution you had *assumed* — `np.random.poisson(lam)` — and asked what worlds it would make.
Here you assume nothing at all and draw from the data itself.

### ✏️ Your turn 2

Bootstrap the slope, and turn the answer into a range.

Seed with `np.random.seed(88)`. Then, 2,000 times: draw `sea.sample(len(sea), replace=True)`, fit
`LinearRegression` to that resample as you have twice already, and append `.coef_[0]` to a list.
Make the finished list an array with `np.array(...)` so you can do arithmetic on it.

Then read the middle 95% off it: `np.percentile(slopes, [2.5, 97.5])` returns the two values that
cut off the bottom 2.5% and the top 2.5%. Print the slope, the two ends, and how many of your
2,000 slopes came out at zero or below — `(slopes <= 0).sum()`.

**Use these names**, because the self-check looks for them: `slopes`, `ci_low`, `ci_high`.

In [ ]:
# ← your answer here


assert slopes.min() > 0, \
    "no resample of this record makes the Bay flat — check you fitted sea_mm against year"
assert ci_low < slope < ci_high, "the interval should straddle the slope you started from"
print("✓ the slope, with an interval —", round(slope, 2), "mm/yr, 95% interval",
      round(ci_low, 2), "to", round(ci_high, 2))

That range is a **confidence interval**: *Not one number but the range your number would have
wandered over, had the world rolled differently.* Written the way a paper would write it:
1.96 mm/yr, 95% CI [1.88, 2.05].

Every one of those 2,000 slopes is worth drawing, because the interval is just two cuts through a
shape.

In [ ]:
plt.hist(slopes, bins=40)
plt.axvline(ci_low, color="firebrick")
plt.axvline(ci_high, color="firebrick")
plt.xlabel("bootstrap slope (mm per year)")
plt.ylabel("number of resamples")
plt.title(f"{len(slopes):,} bootstrap slopes, 95% interval marked")
plt.show()

A hump, and the red lines cut 50 resamples off each side. Two things to read
off it. Nothing in it comes anywhere near zero — the count you printed of resamples at zero or
below was 0 out of 2,000 — so the answer to *could the Bay be flat?* is no, and
now we can say so rather than assert it.

Second, look at how narrow it is: 0.17 mm/yr from end to end, against the
4.03 mm/yr that separated the highest of your five chunks from the lowest. It is
tempting to read that as the chunks having been noise all along — and it is the exact mistake this
week exists to prevent. The two numbers are not the same kind of thing. 0.17 is a
95% interval; 4.03 is the largest of five point estimates minus the smallest.
Setting one beside the other settles nothing, in either direction.

What would settle it is giving the chunks intervals of their own, and that is the loop you have
just written, run on a slice of `sea` instead of all of it. Do it for the two ends.

In [ ]:
for start in [1900, 2000]:
    chunk = sea[(sea["year"] >= start) & (sea["year"] < start + 25)]

    np.random.seed(88)
    chunk_boot = []
    for i in range(2000):
        boot = chunk.sample(len(chunk), replace=True)
        chunk_boot.append(LinearRegression().fit(boot[["year"]], boot["sea_mm"]).coef_[0])

    low, high = np.percentile(chunk_boot, [2.5, 97.5])
    print(start, "-", start + 25, ": 95% interval", round(low, 2), "to", round(high, 2), "mm/yr")

[-0.73, 0.93] for the first quarter-century and
[3.14, 5.12] for the last. They do not overlap: there
are 2.21 mm/yr of daylight between the top of one and the bottom of the other.

So both halves of the story are true and only one of them was worth saying. Short windows really
are wobbly — the first chunk's interval is 1.66 mm/yr wide, nearly
10 times the whole record's 0.17, because a quarter of the data
has to fit through a quarter-century of weather. But that wobble is nowhere near large enough to
open a gap of 2.21. Something in this record really did change, and
1.965 mm/yr is an average across a rate that was not constant. The interval around it
says how well we know that average — not that the Bay rose steadily.

## How much water is that by 2100?

Millimetres per year is not a quantity anyone plans a seawall around. Two conversions get asked
for, they are not the same number, and they are confused constantly:

- **per century** — how much the water rises in a hundred years at this rate.
- **from the end of this record to 2100** — which is a shorter stretch, so it is a smaller number.

Whichever you quote, the interval comes with it: convert `ci_low` and `ci_high` exactly as you
convert the slope, and the answer stays a range.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
fit = LinearRegression().fit(sea[["year"]], sea["sea_mm"])
slope = fit.coef_[0]

np.random.seed(88)
slopes = []
for i in range(2000):
    boot = sea.sample(len(sea), replace=True)
    slopes.append(LinearRegression().fit(boot[["year"]], boot["sea_mm"]).coef_[0])

slopes = np.array(slopes)
ci_low, ci_high = np.percentile(slopes, [2.5, 97.5])

### ✏️ Your turn 3

Report the rise both ways, each as a point estimate and a 95% interval, in **centimetres**.

There are 10 millimetres in a centimetre. The record ends at the close of 2025, so
2100 is 75 years away — not 100.

Print two lines: the rise per century, and the rise from the end of the record to 2100, each with
its interval. Do the conversion on `ci_low` and `ci_high` as well as on `slope`.

**Use these names**, because the self-check looks for them: `cm_century`, `cm_2100`.

In [ ]:
# ← your answer here


assert cm_2100 < cm_century, "2100 is closer than a century away, so it must be the smaller number"
print("✓ the same slope, two questions —", round(cm_century, 1),
      "cm per century, but", round(cm_2100, 1), "cm between the end of the record and 2100")

19.6 cm per century and 14.7 cm by 2100 are the same slope answering
two different questions, and quoting one for the other is a 5-centimetre
mistake.

Be careful about what the second number is. It is what the water does **if the rise continues at
the average rate of the last 126 years**, and the chunk slopes you computed give a
plain reason to doubt that: the most recent twenty-five years came out at 4.1
mm/yr, more than twice the long-run figure. The interval [14.1,
15.4] cm is the uncertainty in *this straight line*, not the uncertainty in what
the ocean will do. It is a floor, not a forecast.

## Was the forecast wrong, or were we unlucky?

Back to the earthquakes, with the same six lines.

`quakes` is every California earthquake between magnitude 3.5 and 6.9 recorded in thirty-six
years — 6,282 of them, in the same latitude-longitude box you used before. The
Gutenberg-Richter recipe is the one you already know: count how many events reached each
magnitude, plot those counts on a log axis against magnitude, fit a straight line, and read the
line off at a magnitude you have not observed.

Written as a function it is short enough to call 2,000 times.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
quakes = load(FDSN + "&starttime=1990-01-01&endtime=2026-01-01"
                     "&minmagnitude=3.5&maxmagnitude=6.9" + CA_BOX,
              "week08_ca_1990_2026_M3.5-6.9.csv")

In [ ]:
edges = np.arange(4.0, 5.6, 0.1).round(1)      # fit between magnitude 4.0 and 5.5


def predicted_m7(mags):
    """Fit Gutenberg-Richter to these magnitudes and read off the expected number of M7+."""
    counts = []
    for edge in edges:
        counts.append((mags >= edge).sum())
    line = LinearRegression().fit(edges.reshape(-1, 1), np.log10(counts))
    return 10 ** line.predict([[7.0]])[0]


print("events at magnitude 4.0 and above:", (quakes["mag"] >= 4.0).sum())
print("events at magnitude 5.5 and above:", (quakes["mag"] >= 5.5).sum())
print("expected number of M7+ in the same span:", round(predicted_m7(quakes["mag"]), 2))

1.64 expected, against five that actually happened. That is where the argument stopped
when you first fitted the line.

### Predict before you run

Bootstrap that 1.64 the way you just bootstrapped the slope and you get a 95% interval
around it. **How high do you think the top of that interval reaches?** Commit to a number — change
`my_guess` and run it — and then find out.

In [ ]:
my_guess = 3.0

print("you guessed the interval reaches:", my_guess, "M7+ earthquakes")
print("five actually happened")

### ✏️ Your turn 4

Bootstrap the forecast. This is the loop you wrote for the slope, with one line changed.

Seed with `np.random.seed(88)`. Then, 2,000 times: draw
`quakes["mag"].sample(len(quakes), replace=True)`, pass it to `predicted_m7`, and append the
answer. Make the list an array, then take `np.percentile(boot_rates, [2.5, 97.5])`.

Print the forecast and its interval.

**Use these names**, because the self-check looks for them: `boot_rates`, `rate_low`, `rate_high`.

In [ ]:
# ← your answer here


assert rate_low < predicted_m7(quakes["mag"]) < rate_high, \
    "the interval should straddle the forecast — check you resampled inside the loop"
assert rate_high < 5, "if your interval reaches five, predicted_m7 was fed something else"
print("✓ the forecast, with an interval —", round(rate_low, 2), "to",
      round(rate_high, 2), "M7+ earthquakes, against 5 observed")

In [ ]:
plt.hist(boot_rates, bins=40)
plt.axvline(rate_high, color="firebrick")
plt.axvline(5, color="black")
plt.xlabel("bootstrap forecast (M7+ earthquakes in 36 years)")
plt.ylabel("number of resamples")
plt.title(f"{len(boot_rates):,} bootstrap forecasts; interval top red, 5 observed black")
plt.show()

The interval runs [0.98, 2.51], and the black line at 5 is off past
the end of everything. Not one of the 2,000 resampled catalogues produced a forecast anywhere near
five. Case closed?

No — and this is the most important paragraph in the week. That interval answers the question
*how well do we know the average rate?* Five is not an average rate. Five is a **count**, one
draw of a thing that scatters even when the rate is exactly right, and you have already simulated
exactly that scatter: `np.random.poisson(lam)` turns a rate into the number of events an
individual thirty-six years actually delivers. Comparing a count to an interval on a rate is
comparing two different quantities.

So put the two sources of wobble together. Take each of your 2,000 bootstrapped rates, and let the
world roll once at that rate.

### ✏️ Your turn 5

Turn the interval on the rate into an interval on the count.

`np.random.poisson(boot_rates)` draws one Poisson count for every rate in the array at once, so
this needs no loop: seed with `np.random.seed(88)`, make `boot_counts`, and take
`np.percentile(boot_counts, [2.5, 97.5])`.

Then print the fraction of those simulated worlds that delivered five or more —
`(boot_counts >= 5).mean()`. That fraction is the answer to *were we unlucky?*

**Use these names**, because the self-check looks for them: `boot_counts`.

In [ ]:
# ← your answer here


assert boot_counts.max() >= 5, \
    "a Poisson draw reaches five now and then; rounding the rates off never would"
assert (boot_counts == 0).sum() > 0, \
    "and it delivers empty thirty-six-year stretches too — check you drew from Poisson"
print("✓ an interval on the count — 5 or more happened in",
      round(100 * (boot_counts >= 5).mean(), 1), "% of simulated worlds")

In [ ]:
plt.hist(boot_counts, bins=np.arange(-0.5, boot_counts.max() + 1.5, 1))
plt.axvline(5, color="black")
plt.xlabel("M7+ earthquakes in a simulated 36 years")
plt.ylabel("number of simulated worlds")
plt.title(f"{len(boot_counts):,} simulated worlds, 5 observed marked in black")
plt.show()

The picture is completely different. Once the counting noise is in, worlds with five **or more**
large earthquakes do turn up: 2.9% of them, and the busiest reached
7. The 95% interval on the count is [0, 5], and five
is sitting on its upper edge rather than outside it.

Read that percentage carefully, because the "or more" is doing work. It is the fraction of worlds
that reached *at least* five, which is what you want when you are asking whether an observation is
extreme — the worlds that landed on exactly five are a smaller share again. A tail is always
counted outward from the observation, never at it.

So the honest verdict on this window is *unlikely, not impossible* — five or more is a
one-in-34 outcome. That is the kind of result that should send you to look at more data rather than to a
conclusion. And there is more data: thirty-six years is not the only thirty-six years California
has had.

In [ ]:
big["when"] = pd.to_datetime(big["time"])
print(big[["mag", "place"]])
print("days from each one to the next:", sorted(big["when"].diff().dropna().dt.days))

window_counts = []
for start in [1918, 1954, 1990]:
    inside = (big["when"] >= f"{start}-01-01") & (big["when"] < f"{start + 36}-01-01")
    window_counts.append(inside.sum())
    print(start, "-", start + 36, ":", inside.sum(), "M7+ earthquakes")

1, 2, 5. The forecast of 1.64
per 36 years is an excellent description of the two earlier windows and a poor one of
the most recent. So the last cell: if the rate really is what Gutenberg-Richter says, and it held
for all 108 years, how often do you get 8 or more in total?

In [ ]:
np.random.seed(88)
long_counts = np.random.poisson(boot_rates * 3)

print("expected over", 108, "years:", round(3 * predicted_m7(quakes["mag"]), 2))
print("observed:", sum(window_counts))
print("fraction of simulated worlds reaching that:",
      round((long_counts >= sum(window_counts)).mean(), 3))

8 against an expected 4.9, and 16%
of simulated 108-year worlds do at least that well. On the long record the forecast
is not in trouble at all.

### ✏️ Your turn 6

One last number out of that rate — and this time the interval comes with it from the start.

In the probability week you counted earthquakes near campus, divided by the years to get a rate,
and turned the rate into a chance of at least one with `1 - np.exp(-rate * years)`. You reported
that chance as a single number, because a single rate was all you had. You now have 2,000 rates.

`boot_rates` counts M7+ earthquakes per 36 years, so `boot_rates / 36` is
a rate per year, and `1 - np.exp(-boot_rates / 36 * 30)` is the chance of at
least one M7+ inside the box in the next 30 years — all 2,000 of them at once, no loop
needed.

Print the chance the class forecast gives, which is the same formula applied to
`predicted_m7(quakes["mag"])`, and then the 95% interval from `np.percentile`.

**Use these names**, because the self-check looks for them: `boot_probs`, `p_low`, `p_high`.

In [ ]:
# ← your answer here


assert p_low < 1 - np.exp(-predicted_m7(quakes["mag"]) / 36 * 30) < p_high, \
    "the interval should straddle the chance the class forecast gives"
assert boot_probs.max() < 0.99, \
    "divide the rate by 36 before multiplying it by 30"
print("✓ a probability with an interval — between", round(100 * p_low), "and",
      round(100 * p_high), "% chance of at least one M7+ in the next 30 years")

## What did we assume without saying so?

Four caveats you should carry, because none of them is settled by anything above. The box is a
rectangle of latitude and longitude, not the state, so it collects earthquakes in Nevada and Baja
California as well: of the 8 large earthquakes listed above, Fairview Peak is in
Nevada and Sierra El Mayor is in Baja California. And the earlier windows
depend on a catalogue that was thinner: *A catalogue lists what somebody's instruments recorded,
not what happened.* If those windows are undercounted, the true long-run rate is higher than
8 in 108 years, and that pushes the same way — it makes five look less
exceptional, not more.

The third is an assumption every `np.random.poisson` above made without saying so: that
large earthquakes arrive **independently**, one roll of the dice each. The gaps you printed say
they do not. The shortest is 63 days — 1992 Petrolia in April and
1992 Landers in June, two months apart in a record whose typical gap is years — and the three
window counts scatter more than a Poisson process allows, their variance
1.63 times their mean where Poisson holds the two equal. Clustered events make a
count scatter *more* than Poisson, so the true share of thirty-six-year worlds reaching five or
more is **larger** than the 2.9% you computed, not smaller. That is the same
direction as the caveat before it, and it leaves the verdict standing — but it is an assumption,
it is visible in our own table, and it should have been said out loud.

The fourth is the one that moves the numbers, and it is hiding inside the words *magnitude 7*. A
magnitude is a measurement with an uncertainty of a tenth or two, and 7.0 is a round number
somebody chose, not a boundary in the rock. So try the number on the other side of it. Re-run the
same query with the threshold at 6.9 — still an earthquake any seismologist would describe as
roughly magnitude 7 — and count the same three windows again.

In [ ]:
big69 = load(FDSN + "&starttime=1918-01-01&endtime=2026-01-01&minmagnitude=6.9" + CA_BOX,
             "week08_ca_1918_2026_M6.9.csv")
big69["when"] = pd.to_datetime(big69["time"])
print("days from each one to the next:", sorted(big69["when"].diff().dropna().dt.days))

for start in [1918, 1954, 1990]:
    inside = (big69["when"] >= f"{start}-01-01") & (big69["when"] < f"{start + 36}-01-01")
    print(start, "-", start + 36, ":", inside.sum(), "M6.9+ earthquakes")

3, 4, 5 — against
1, 2, 5 a moment ago. One tenth of a
magnitude adds 4 earthquakes (1927 Lompoc, 1940 Imperial Valley, 1954 Dixie Valley, 1989 Loma Prieta), every one of them in an
earlier window. The counts are still rising and the most recent window is still the busiest, so
the excess has not vanished — it has **shrunk**, from 3.3 times the average of the
two earlier windows to 1.4 times. The difference matters: 3.3
times is the sort of gap that starts an argument, and 1.4 times is what Poisson
scatter around a mean of 4 hands you routinely.

One of the 4 is worth a second look on its own. 1954 Dixie Valley is the zero in
that list of gaps: it follows 1954 Fairview Peak by 4 minutes, and the two are
counted here as two independent events. That is the third caveat made concrete.

So *1, 2, 5* was never a fact about
California on its own. It was a fact about California **and a threshold**, and the threshold moved
the story further than the earthquakes did. That is the same lesson as the interval, applied to a
choice instead of a sample: when a conclusion rests on one round number, try the number either
side of it and report what you find.

Back to the Bay for the assumption the bootstrap slipped past you. Resampling *months* treats each
month as an independent draw — as if the sea level in March told you nothing about April. It
plainly does. A wet winter, an El Niño, a warm year: those last longer than a month, so
neighbouring months carry much of the same information, and the record's months are worth rather
less than that many independent measurements.

The fix is to resample bigger pieces. Draw whole **calendar years** with replacement — all twelve
months of 1931 or none of them — so that whatever is shared inside a year travels with it. There
are 126 years to draw from. `pd.concat` is the function that stacks the chosen
years back into one table.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
np.random.seed(88)
slopes = []
for i in range(2000):
    boot = sea.sample(len(sea), replace=True)
    slopes.append(LinearRegression().fit(boot[["year"]], boot["sea_mm"]).coef_[0])

slopes = np.array(slopes)
ci_low, ci_high = np.percentile(slopes, [2.5, 97.5])

In [ ]:
np.random.seed(88)
by_year = {}
for y, one_year in sea.groupby("Year"):
    by_year[y] = one_year

block_slopes = []
for i in range(2000):
    picked = np.random.choice(sorted(by_year), size=len(by_year), replace=True)
    boot = pd.concat([by_year[y] for y in picked])
    block_slopes.append(LinearRegression().fit(boot[["year"]], boot["sea_mm"]).coef_[0])

block_slopes = np.array(block_slopes)
block_low, block_high = np.percentile(block_slopes, [2.5, 97.5])

print("resampling months:", round(ci_low, 2), "to", round(ci_high, 2), "mm/yr")
print("resampling years: ", round(block_low, 2), "to", round(block_high, 2), "mm/yr")

In [ ]:
bins = np.arange(min(slopes.min(), block_slopes.min()),               # ONE set of bins for both,
                 max(slopes.max(), block_slopes.max()) + 0.01, 0.01)  # or the bar widths compare

plt.hist(slopes, bins=bins, label="months resampled")
plt.hist(block_slopes, bins=bins, alpha=0.6, label="whole years resampled")
plt.xlabel("bootstrap slope (mm per year)")
plt.ylabel("number of resamples")
plt.title(f"{len(slopes):,} resamples each, the same record")
plt.legend()
plt.show()

The interval got **wider** — [1.79, 2.14] against
[1.88, 2.05], about 2.0 times the width — and that is
the direction nobody guesses. Respecting the structure of your data does not sharpen the answer;
it stops you from claiming a sharpness you never had.

The month-by-month interval was too narrow because it counted every month as a fresh piece of
information. How many were there really? An interval 2.0 times wider is
precisely the interval you would get from 382 independent months, so that is
what this record is worth. And note that 382 is *not*
126: 126 is how many blocks the loop drew, which is the
method, not the result.

Zero is still nowhere near either interval, so the conclusion about the Bay survives. That will
not always be true, and when it is not, the wider interval is the one to believe.

## The question, answered

**We cannot show it was wrong — and the reason we can now say that, rather than shrug, is that
the forecast finally has an interval attached.** Bootstrapping the catalogue puts the
Gutenberg-Richter rate at 1.64 M7+ earthquakes per 36 years, 95% CI
[0.98, 2.51], and five is far outside that. But a rate is not a
count: fold in the Poisson scatter that any real thirty-six years is subject to and five or more
happens in 2.9% of simulated worlds, sitting on the top edge of the interval
on the count rather than beyond it. Widen the window and the case for a broken model gets weaker still — 8 large
earthquakes in 108 years against 4.9 expected, which
16% of simulated worlds match or beat. The forecast was not convicted; it
was not acquitted either, and the honest report is the interval rather than the verdict. Written
that way it is still usable: the same rate says the chance of at least one M7+ in the box in the
next 30 years is 75%, 95% CI [56%,
88%].

## Week 8 summary

**The question.** Was our earthquake forecast wrong — or were we just unlucky?

### What to remember

| | |
|---|---|
| **1** | A single number is not an answer. Report an interval. |
| **2** | Resampling your own data tells you how much your estimate would have wobbled. |
| **3** | An interval tells you what would have wobbled; check you are comparing like with like before you call a model broken. |

### The ideas, in plain words

| Idea | Means |
|---|---|
| **Bootstrap** | Ask the data the same question a thousand times, using a different random slice of itself each time. |
| **Confidence interval** | Not one number but the range your number would have wandered over, had the world rolled differently. |

### Code you met this week

| Function | What it does |
|---|---|
| `table.sample(n, replace=True)` | draw a new table the same size, picking rows at random and putting each back |
| `np.percentile(values, [2.5, 97.5])` | the two values that cut off the bottom and top 2.5% — a 95% interval |

## Homework

Three parts on the two datasets you already have loaded. Part 1 goes back to the Bay and asks
something class did not; parts 2 and 3 make you re-run the earthquake argument on a fitting range
you choose yourself. If you have restarted since class, run the setup cell at the top first, then
the checkpoint just below.

In [ ]:
# ── Checkpoint ── run this if you restarted the kernel or fell behind ──
edges = np.arange(4.0, 5.6, 0.1).round(1)


def predicted_m7(mags):
    """Fit Gutenberg-Richter to these magnitudes and read off the expected number of M7+."""
    counts = []
    for edge in edges:
        counts.append((mags >= edge).sum())
    line = LinearRegression().fit(edges.reshape(-1, 1), np.log10(counts))
    return 10 ** line.predict([[7.0]])[0]

### ✏️ Your turn 7

In class the first and the last twenty-five years came out with intervals that do not overlap —
[-0.73, 0.93] against
[3.14, 5.12] — which is how we concluded that the rise
had really changed. Those were the two extremes of five. Ask the same question of a cut that uses
every month: does the conclusion survive?

Split the record in half — 1900 up to 1963, and 1963 up to
2026 — and bootstrap each half separately, exactly as you bootstrapped the whole record
in your turn 2. Print each half's slope and 95% interval.

Then print whether the two intervals overlap. Two intervals overlap when the lower one's top end is
above the higher one's bottom end, so `first_high > second_low` is the test. Report what you get,
whichever way it comes out.

Resample **months**, as your turn 2 did, not whole years. Class showed that months give an
interval that is too narrow — and too narrow is the direction that makes two intervals *miss* each
other, so if these two overlap anyway, the wider honest version would only overlap more.

Then, as a comment at the end of your cell, answer in one sentence from your own two intervals:
does the conclusion that the rise really changed survive a cut that uses every month, or was it an
artefact of comparing the two extremes of five?

**Use these names**, because the self-check looks for them: `first`, `second`, `first_slope`,
`first_low`, `first_high`, `second_slope`, `second_low`, `second_high`.

In [ ]:
# ← your answer here


assert len(first) + len(second) == len(sea), "every month belongs to exactly one half"
assert first_low < first_slope < first_high, \
    "the first half's interval should straddle the first half's own slope"
assert second_low < second_slope < second_high, \
    "and the second half's should straddle the second half's"
print("✓ the two halves — first", round(first_low, 2), "to", round(first_high, 2),
      ", second", round(second_low, 2), "to", round(second_high, 2),
      "; overlapping:", first_high > second_low)

### ✏️ Your turn 8

Class fit Gutenberg-Richter between magnitude 4.0 and 5.5. That was a choice, and it was not the
only defensible one — too low and the catalogue is missing small events, too high and there are
barely any events left to fit.

**Pick one:** fit between 3.5 and 5.0, or between 4.5 and 6.0. Change
one line — `edges = np.arange(...).round(1)` — and re-run the whole argument on your choice:
`predicted_m7`, the 2,000 bootstrap rates, the Poisson draw, and the fraction of simulated worlds
reaching five. Redefining `edges` is enough: `predicted_m7` reads it, so nothing else needs editing.

Print your forecast, your 95% interval on the count, and your fraction. Then, as a comment at the
end of your cell, say in one sentence why moving the fitting range moves the forecast at all —
what changes about the line when you fit lower or higher.

**Use these names**, because the self-check looks for them: `edges`, `my_counts`, `my_fraction`.

In [ ]:
# ← your answer here


assert edges[0] != 4.0 or edges[-1] != 5.5, "change the range — this is class's fit, not a choice"
assert 0 < my_fraction < 0.2, \
    "a small but nonzero share should reach five — zero means the Poisson draw is missing"
print("✓ your own fitting range — magnitude", edges[0], "to", edges[-1],
      ", 5 or more reached in", round(100 * my_fraction, 1), "% of simulated worlds")

### ✏️ Your turn 9

Two or three sentences, quoting your own numbers from your turn 8.

Class's fitting range put five large earthquakes at the top edge of the interval. Say what your
range put it at, and whether your choice changes the verdict — whether a reader of your notebook
would come away thinking the forecast is broken, or thinking the last thirty-six years were busy.
Then say what would have to be true for a single thirty-six-year count to settle the question
either way.

*(Double-click this cell and replace this line with your answer.)*